In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from sklearn.model_selection import train_test_split

from customer_intelligence_churn_prediction_system.data.load_data import (
    load_raw_data
)

from customer_intelligence_churn_prediction_system.preprocessing.clean_data import (
    clean_customer_data
)

from customer_intelligence_churn_prediction_system.features.build_features import (
    build_customer_features
)
from customer_intelligence_churn_prediction_system.models.evaluate_models import (
    evaluate_model
)

In [2]:

df = load_raw_data()

cleaned_df = clean_customer_data(df)

featured_df = build_customer_features(cleaned_df)

print("Featured dataset shape:", featured_df.shape)

Featured dataset shape: (7043, 25)


In [3]:

X = featured_df.drop(columns=["customerID", "Churn"])

y = featured_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 23)
y shape: (7043,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (5634, 23)
Testing shape: (1409, 23)


In [5]:
from customer_intelligence_churn_prediction_system.preprocessing.preprocess import (
    create_preprocessor
)

from customer_intelligence_churn_prediction_system.models.train_models import (
    build_logistic_regression_pipeline
)

preprocessor = create_preprocessor(X_train)

logistic_model = build_logistic_regression_pipeline(preprocessor)

logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed!")

Logistic Regression training completed!


In [6]:
logistic_metrics = evaluate_model(
    logistic_model,
    X_test,
    y_test
)

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Confusion Matrix:
[[932 103]
 [179 195]]


In [7]:
from customer_intelligence_churn_prediction_system.models.train_models import (
    build_random_forest_pipeline
)

rf_preprocessor = create_preprocessor(X_train)

random_forest_model = build_random_forest_pipeline(
    rf_preprocessor
)

random_forest_model.fit(X_train, y_train)

print("Random Forest training completed!")

Random Forest training completed!


In [8]:
rf_metrics = evaluate_model(
    random_forest_model,
    X_test,
    y_test
)

print("\nRandom Forest Metrics:")

for metric, value in rf_metrics.items():
    print(f"{metric}: {value:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.61      0.48      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409

Confusion Matrix:
[[920 115]
 [193 181]]

Random Forest Metrics:
Accuracy: 0.7814
Precision: 0.6115
Recall: 0.4840
F1 Score: 0.5403
ROC-AUC: 0.8225


In [9]:
from customer_intelligence_churn_prediction_system.models.train_models import (
    build_xgboost_pipeline
)

xgb_preprocessor = create_preprocessor(X_train)

xgboost_model = build_xgboost_pipeline(xgb_preprocessor)

xgboost_model.fit(X_train, y_train)

print("XGBoost training completed!")

XGBoost training completed!


In [10]:
xgb_metrics = evaluate_model(
    xgboost_model,
    X_test,
    y_test
)

print("\nXGBoost Metrics:")

for metric, value in xgb_metrics.items():
    print(f"{metric}: {value:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Confusion Matrix:
[[934 101]
 [183 191]]

XGBoost Metrics:
Accuracy: 0.7984
Precision: 0.6541
Recall: 0.5107
F1 Score: 0.5736
ROC-AUC: 0.8425


In [11]:
from customer_intelligence_churn_prediction_system.models.train_models import (
    build_lightgbm_pipeline
)

lgbm_preprocessor = create_preprocessor(X_train)
lightgbm_model = build_lightgbm_pipeline(lgbm_preprocessor)

lightgbm_model.fit(X_train, y_train)

print("LightGBM training completed!")

LightGBM training completed!


In [12]:
lgbm_metrics = evaluate_model(
    lightgbm_model,
    X_test,
    y_test
)

print("\nLightGBM Metrics:")

for metric, value in lgbm_metrics.items():
    print(f"{metric}: {value:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.53      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409

Confusion Matrix:
[[920 115]
 [176 198]]

LightGBM Metrics:
Accuracy: 0.7935
Precision: 0.6326
Recall: 0.5294
F1 Score: 0.5764
ROC-AUC: 0.8334


In [13]:
model_comparison = pd.DataFrame([
    {"Model": "Logistic Regression", **logistic_metrics},
    {"Model": "Random Forest", **rf_metrics},
    {"Model": "XGBoost", **xgb_metrics},
    {"Model":"LightGBM", **lgbm_metrics}
])

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.799858,0.654362,0.521390,0.580357,0.842375
1,Random Forest,0.781405,0.611486,0.483957,0.540299,0.822530
2,XGBoost,0.798439,0.654110,0.510695,0.573574,0.842453
3,LightGBM,0.793471,0.632588,0.529412,0.576419,0.833450


## Baseline Model Comparison — Findings

Four baseline classification models were evaluated:
1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM

Key observations:
- Logistic Regression achieved the best overall F1-score (0.5804).
- XGBoost achieved the highest ROC-AUC (0.8425).
- LightGBM achieved the highest churn recall (0.5294).
- Random Forest showed the weakest baseline performance.

Conclusion:
Logistic Regression is currently the strongest overall baseline.
However, no final model has been selected. Further optimization
and evaluation are required.

In [14]:
from sklearn.model_selection import RandomizedSearchCV,StratifiedKFold

In [15]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle  = True,
    random_state=42
)

In [17]:
from sklearn.model_selection import RandomizedSearchCV

lr_preprocessor = create_preprocessor(X_train)

lr_pipeline = build_logistic_regression_pipeline(
    lr_preprocessor
)

lr_param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"]
}

lr_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_param_grid,
    n_iter=10,
    scoring="f1",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)

print("Best Parameters:", lr_search.best_params_)
print("Best Cross-Validation F1:", lr_search.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters: {'classifier__class_weight': 'balanced', 'classifier__C': 10}
Best Cross-Validation F1: 0.6294014313394092


In [18]:
best_lr_model = lr_search.best_estimator_

In [19]:
rf_preprocessor = create_preprocessor(X_train)

rf_pipeline = build_random_forest_pipeline(
    rf_preprocessor
)

rf_param_grid = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [None, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__class_weight": [None, "balanced", "balanced_subsample"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=15,
    scoring="f1",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

best_rf_model = rf_search.best_estimator_

print("Best Parameters:", rf_search.best_params_)
print("Best CV F1 Score:", rf_search.best_score_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 20, 'classifier__class_weight': 'balanced'}
Best CV F1 Score: 0.6331799473221491


In [20]:
xgb_preprocessor = create_preprocessor(X_train)

xgb_pipeline = build_xgboost_pipeline(
    xgb_preprocessor
)

xgb_param_grid = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [3, 4, 5, 7],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__scale_pos_weight": [1, 2, 3]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=20,
    scoring="f1",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

best_xgb_model = xgb_search.best_estimator_

print("Best Parameters:", xgb_search.best_params_)
print("Best CV F1 Score:", xgb_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'classifier__subsample': 0.7, 'classifier__scale_pos_weight': 2, 'classifier__n_estimators': 500, 'classifier__min_child_weight': 5, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.01, 'classifier__colsample_bytree': 0.9}
Best CV F1 Score: 0.632336546471368


In [21]:
lgbm_preprocessor = create_preprocessor(X_train)

lgbm_pipeline = build_lightgbm_pipeline(
    lgbm_preprocessor
)

lgbm_param_grid = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__num_leaves": [15, 31, 50, 70],
    "classifier__max_depth": [-1, 5, 10, 15],
    "classifier__min_child_samples": [10, 20, 30, 50],
    "classifier__class_weight": [None, "balanced"],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=lgbm_param_grid,
    n_iter=20,
    scoring="f1",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)

best_lgbm_model = lgbm_search.best_estimator_

print("Best Parameters:", lgbm_search.best_params_)
print("Best CV F1 Score:", lgbm_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'classifier__num_leaves': 15, 'classifier__n_estimators': 500, 'classifier__min_child_samples': 20, 'classifier__max_depth': -1, 'classifier__learning_rate': 0.01, 'classifier__colsample_bytree': 0.7, 'classifier__class_weight': 'balanced'}
Best CV F1 Score: 0.6341543976640118


In [22]:
tuned_models = {
    "Logistic Regression": best_lr_model,
    "Random Forest": best_rf_model,
    "XGBoost": best_xgb_model,
    "LightGBM": best_lgbm_model
}

In [23]:
from customer_intelligence_churn_prediction_system.models.evaluate_models import evaluate_model
import pandas as pd

test_results = {}

for model_name, model in tuned_models.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print('='*60)

    metrics = evaluate_model(model, X_test, y_test)
    test_results[model_name] = metrics

results_df = pd.DataFrame(test_results).T
results_df = results_df.sort_values(by="F1 Score", ascending=False)

results_df


Evaluating: Logistic Regression
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.71      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.73      0.75      1409

Confusion Matrix:
[[739 296]
 [ 81 293]]

Evaluating: Random Forest
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.76      0.83      1035
           1       0.54      0.77      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.77      0.78      1409

Confusion Matrix:
[[791 244]
 [ 87 287]]

Evaluating: XGBoost
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.79      0.84      1035
     

,Accuracy,Precision,Recall,F1 Score,ROC-AUC
LightGBM,0.758694,0.530035,0.802139,0.638298,0.847109
Random Forest,0.765082,0.540490,0.767380,0.634254,0.838296
XGBoost,0.773598,0.556936,0.719251,0.627771,0.846543
Logistic Regression,0.732434,0.497453,0.783422,0.608515,0.841050


In [24]:
best_model = best_lgbm_model